# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FawadAhmad-bilal/flyrank-assignment-1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{token}')")

rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance"

con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM read_parquet('{rel}/**/*.parquet', hive_partitioning=1)
    WHERE month = '2026-03'
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   n   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/**/*.parquet', hive_partitioning=1) LIMIT 1").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
con.sql(f"""
    SELECT COUNT(*), MIN(report_date), MAX(report_date)
    FROM read_parquet('{rel}/**/*.parquet', hive_partitioning=1)
    WHERE month = '2026-03'
""").show()

con.sql(f"""
    SELECT COUNT(*)
    FROM read_parquet('{rel}/**/*.parquet', hive_partitioning=1)
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
""").show()

con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS null_impr,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS null_clicks
    FROM read_parquet('{rel}/**/*.parquet', hive_partitioning=1)
    WHERE month = '2026-03'
""").show()

┌──────────────┬──────────────────┬──────────────────┐
│ count_star() │ min(report_date) │ max(report_date) │
│    int64     │       date       │       date       │
├──────────────┼──────────────────┼──────────────────┤
│      9841378 │ 2026-03-01       │ 2026-03-31       │
└──────────────┴──────────────────┴──────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3611061 │
└──────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬─────────────┐
│ null_impr │ null_clicks │
│   int64   │    int64    │
├───────────┼─────────────┤
│         0 │           0 │
└───────────┴─────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [6]:
from datasets import load_dataset
import pandas as pd

dim_clients_df = load_dataset(
    "FlyRank/internship-warehouse", "dim_clients", split="train"
).to_pandas()

con.register("dim_clients", dim_clients_df)

con.sql(f"""
    SELECT
        c.gsc_data_start,
        COUNT(*) AS n_content_rows
    FROM read_parquet('{rel}/**/*.parquet', hive_partitioning=1) f
    JOIN dim_clients c
      ON f.client_hash_id = c.client_hash_id
    WHERE f.month = '2026-03'
    GROUP BY c.gsc_data_start
    ORDER BY c.gsc_data_start
""").show()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_clients.parquet: reconstructing file:   0%|          |  0.00B / 3.38kB            

dim_clients.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/104 [00:00<?, ? examples/s]

┌────────────────┬────────────────┐
│ gsc_data_start │ n_content_rows │
│      date      │     int64      │
├────────────────┼────────────────┤
│ 2025-01-27     │         235538 │
│ 2025-02-11     │         869640 │
│ 2025-03-11     │         335379 │
│ 2025-06-07     │         756660 │
│ 2025-06-18     │         126512 │
│ 2025-06-21     │         484091 │
│ 2025-06-29     │         316610 │
│ 2025-07-01     │         988497 │
│ 2025-07-06     │          36538 │
│ 2025-07-07     │          53165 │
│     ·          │            ·   │
│     ·          │            ·   │
│     ·          │            ·   │
│ 2026-01-09     │         182218 │
│ 2026-02-17     │          12714 │
│ 2026-02-19     │        1284576 │
│ 2026-02-26     │           6766 │
│ 2026-03-11     │           8823 │
│ 2026-03-12     │          19440 │
│ 2026-03-19     │            520 │
│ 2026-03-25     │           1812 │
│ 2026-03-27     │           1216 │
│ NULL           │         219077 │
├────────────────┴──────────

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.